# Autoencoder Conv2D, DS2OS

Autoencoder secuencial que reconstruye ventanas de W = 10 registros consecutivos de un mismo dispositivo.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [2]:
DATA_PATH = Path.home() / ".cache/kagglehub/datasets/libamariyam/ds2os-dataset/versions/1/DS2OS.csv"
df = pd.read_csv(DATA_PATH)
df['accessedNodeType'] = df['accessedNodeType'].fillna('missing')
df['value'] = df['value'].replace({'true': 1, 'false': 0, 'twenty': 20, 'none': 0})
df['value'] = pd.to_numeric(df['value'], errors='coerce').fillna(0)

# Ventanas por dispositivo (sourceID), ordenadas por timestamp
df['group'] = df['sourceID']
df['order'] = df['timestamp']
df['is_attack'] = (df['normality'] != 'normal').astype(int)
CAT_COLS = ['sourceID', 'sourceAddress', 'sourceType', 'sourceLocation',
            'destinationServiceAddress', 'destinationServiceType', 'destinationLocation',
            'accessedNodeAddress', 'accessedNodeType', 'operation']
X = pd.get_dummies(df.drop(columns=['normality', 'timestamp', 'group', 'order', 'is_attack']), columns=CAT_COLS)
meta = df[['group', 'order', 'is_attack']]
print(X.shape)

(357952, 505)


In [3]:
def build_windows(X, meta, W, stride=1):
    meta = meta.copy()
    meta['pos'] = meta.groupby('group').cumcount()
    meta['grp_size'] = meta.groupby('group')['group'].transform('size')
    order_idx = meta.sort_values(['group', 'order']).index
    X = X.loc[order_idx].reset_index(drop=True)
    meta = meta.loc[order_idx].reset_index(drop=True)

    train_mask = (meta['pos'] < (0.8 * meta['grp_size'])).to_numpy()
    scaler = MinMaxScaler().fit(X.to_numpy()[train_mask])
    Xs = scaler.transform(X.to_numpy()).astype('float32')
    attack_arr = meta['is_attack'].to_numpy()
    group_arr = meta['group'].to_numpy()

    train_windows, test_windows, test_labels = [], [], []
    for gid in pd.unique(group_arr):
        sel = np.where(group_arr == gid)[0]
        is_train = train_mask[sel]
        train_idxs = sel[is_train]
        test_idxs = sel[~is_train]

        n_tr = len(train_idxs) - W + 1
        if n_tr > 0:
            attack_tr = attack_arr[train_idxs]
            cums = np.concatenate([[0], np.cumsum(attack_tr)])
            window_sums = cums[W:] - cums[:-W]
            for i in np.where(window_sums == 0)[0][::stride]:
                train_windows.append(Xs[train_idxs[i:i + W]])

        n_te = len(test_idxs) - W + 1
        if n_te > 0:
            attack_te = attack_arr[test_idxs]
            cums = np.concatenate([[0], np.cumsum(attack_te)])
            window_sums = cums[W:] - cums[:-W]
            for i in range(0, n_te, stride):
                test_windows.append(Xs[test_idxs[i:i + W]])
                test_labels.append(int(window_sums[i] > 0))

    X_train_w = np.array(train_windows) if train_windows else np.empty((0, W, X.shape[1]))
    X_test_w = np.array(test_windows) if test_windows else np.empty((0, W, X.shape[1]))
    y_test_w = np.array(test_labels)
    return X_train_w, X_test_w, y_test_w


W = 10
STRIDE = 5
X_train_w, X_test_w, y_test_w = build_windows(X, meta, W, stride=STRIDE)
print(f"Ventanas train: {X_train_w.shape}, ventanas test: {X_test_w.shape}, positivos test: {y_test_w.sum()}/{len(y_test_w)}")

Ventanas train: (54985, 10, 505), ventanas test: (14195, 10, 505), positivos test: 582/14195


In [4]:
input_dim = X_train_w.shape[2]
bottleneck = 16
X_train_w2 = X_train_w[..., None]
X_test_w2 = X_test_w[..., None]

# GlobalAveragePooling y UpSampling en lugar de Flatten y Dense, que con muchas columnas daba una capa enorme
inputs = keras.Input(shape=(W, input_dim, 1))
x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(inputs)
x = layers.Conv2D(8, (3, 3), activation='relu', padding='same')(x)
x = layers.GlobalAveragePooling2D()(x)
bn = layers.Dense(bottleneck, activation='relu')(x)
x = layers.Dense(8, activation='relu')(bn)
x = layers.Reshape((1, 1, 8))(x)
x = layers.UpSampling2D(size=(W, input_dim))(x)
x = layers.Conv2DTranspose(8, (3, 3), activation='relu', padding='same')(x)
x = layers.Conv2DTranspose(16, (3, 3), activation='relu', padding='same')(x)
outputs = layers.Conv2D(1, (1, 1), activation='sigmoid', padding='same')(x)

autoencoder = keras.Model(inputs, outputs)
autoencoder.compile(optimizer='adam', loss='mse')

history = autoencoder.fit(
    X_train_w2, X_train_w2,
    epochs=100, batch_size=512, validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)],
    verbose=0,
)
print(f"Épocas entrenadas: {len(history.history['loss'])}, val_loss final: {history.history['val_loss'][-1]:.5f}")

Épocas entrenadas: 71, val_loss final: 0.01936


In [5]:
def reconstruction_error(X):
    return np.mean(np.square(X - autoencoder.predict(X, verbose=0)), axis=(1, 2, 3))

threshold = np.percentile(reconstruction_error(X_train_w2), 95)
err_test = reconstruction_error(X_test_w2)
y_pred = (err_test > threshold).astype(int)

print(classification_report(y_test_w, y_pred, target_names=['normal', 'ataque']))
print(confusion_matrix(y_test_w, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test_w, err_test):.4f}")

              precision    recall  f1-score   support

      normal       0.96      1.00      0.98     13613
      ataque       1.00      0.01      0.01       582

    accuracy                           0.96     14195
   macro avg       0.98      0.50      0.50     14195
weighted avg       0.96      0.96      0.94     14195

[[13613     0]
 [  578     4]]
ROC-AUC: 0.4575
